# CA-CFAR SAR Ship Detection on PYNQ-Z2 — working pipeline
The actual end-to-end run: load bitstream -> check DMA -> verify one tile bit-exact -> tile an uploaded SAR image through the FPGA -> floor + land mask -> show detection figure.

**Files needed next to this notebook:** `cfar.bit`, `cfar.hwh`, `cfar_reference.py`, and a SAR image.

## 1. Load the bitstream + check the DMA exists

In [ ]:
import numpy as np, time
from pynq import Overlay, allocate
from PIL import Image
import matplotlib.pyplot as plt
import cfar_reference as C

ol = Overlay('/home/xilinx/jupyter_notebooks/cfar/cfar.bit')
print('bitstream loaded. blocks:', list(ol.ip_dict.keys()))
mm2s = ol.axi_dma_0.sendchannel   # PS -> PL (pixels)
s2mm = ol.axi_dma_0.recvchannel   # PL -> PS (detections)

## 2. Verify ONE tile on hardware (HW must equal software reference)

In [ ]:
img = C.make_test_image(seed=7)
NPIX = C.IMG_W * C.IMG_H
in_buf  = allocate(shape=(NPIX,), dtype=np.uint16)
out_buf = allocate(shape=(NPIX,), dtype=np.uint8)
in_buf[:] = img.flatten()

t0 = time.perf_counter()
s2mm.transfer(out_buf); mm2s.transfer(in_buf); mm2s.wait(); s2mm.wait()
dt_ms = (time.perf_counter()-t0)*1e3
det_hw = (np.asarray(out_buf).reshape(C.IMG_H, C.IMG_W) & 1).astype(np.uint8)
det_sw = C.cfar_detect(img)
del in_buf, out_buf
print(f'round-trip {dt_ms:.2f} ms | HW detections {int(det_hw.sum())}')
print('HW == SW reference:', np.array_equal(det_hw, det_sw))

## 3. Run a FULL uploaded SAR image through the FPGA (tiling pipeline)
Change `IMG_FILE` to the image you uploaded.

In [ ]:
IMG_FILE     = '/home/xilinx/jupyter_notebooks/cfar/sample_complex_ocean.png'  # <-- your image
T, WH        = 32, 3
STRIDE       = T - 2*WH       # overlapping tiles so no ship falls in a seam
FLOOR_FRAC   = 0.45           # absolute floor as fraction of max (clears speckle); or None
FLOOR_P      = 98             # percentile floor used if FLOOR_FRAC is None
USE_LANDMASK = False          # True for coastal/city scenes, False for open ocean

scene = np.asarray(Image.open(IMG_FILE).convert('L'), dtype=float)
h, w = scene.shape
gmin, gmax = scene.min(), max(scene.max(), scene.min()+1)
scaled = ((scene-gmin)/(gmax-gmin)*60000).astype(np.uint16)
floor = int(FLOOR_FRAC*60000) if FLOOR_FRAC is not None else int(np.percentile(scaled, FLOOR_P))

def box_blur(a, k):
    f=a.astype(np.float64); ii=np.pad(np.cumsum(np.cumsum(f,0),1),((1,0),(1,0)))
    H,W=f.shape; r=k//2; I=np.arange(H); J=np.arange(W)
    i0=np.clip(I-r,0,H)[:,None]; i1=np.clip(I+r+1,0,H)[:,None]
    j0=np.clip(J-r,0,W)[None,:]; j1=np.clip(J+r+1,0,W)[None,:]
    return (ii[i1,j1]-ii[i0,j1]-ii[i1,j0]+ii[i0,j0])/((i1-i0)*(j1-j0))

if USE_LANDMASK:
    blur = box_blur(scaled, 101)
    water = blur < np.percentile(blur, 88)
    water = box_blur(water.astype(float), 21) > 0.5
else:
    water = np.ones((h, w), bool)

rs = sorted(set([x for x in range(0, max(1, h-T+1), STRIDE)] + [h-T]))
cs = sorted(set([x for x in range(0, max(1, w-T+1), STRIDE)] + [w-T]))
in_buf  = allocate(shape=(T*T,), dtype=np.uint16)
out_buf = allocate(shape=(T*T,), dtype=np.uint8)
det = np.zeros((h, w), dtype=np.uint8); nt = 0
t0 = time.perf_counter()
for r in rs:
    for c in cs:
        in_buf[:] = scaled[r:r+T, c:c+T].flatten()
        s2mm.transfer(out_buf); mm2s.transfer(in_buf); mm2s.wait(); s2mm.wait()
        det[r:r+T, c:c+T] |= (np.asarray(out_buf).reshape(T, T) & 1)
        nt += 1
dt = time.perf_counter() - t0
del in_buf, out_buf

final = det & (scaled >= floor).astype(np.uint8) & water.astype(np.uint8)
print(f'{nt} tiles | raw {int(det.sum())} -> final {int(final.sum())} detections | {dt:.2f}s on FPGA')

## 4. Show + save the detection figure

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 7))
ax[0].imshow(scene, cmap='gray'); ax[0].set_title('Uploaded SAR scene', fontsize=11); ax[0].axis('off')
ax[1].imshow(scene, cmap='gray')
ys, xs = np.where(final == 1)
ax[1].scatter(xs, ys, s=70, facecolors='none', edgecolors='red', linewidths=1.4)
ax[1].set_title(f'FPGA CA-CFAR detection: {int(final.sum())} target pixels, {nt} tiles', fontsize=11)
ax[1].axis('off')
fig.suptitle('Hardware CA-CFAR on PYNQ-Z2 (Zynq-7020 PL)', fontsize=12, y=1.02, weight='bold')
plt.tight_layout()
fig.savefig('/home/xilinx/jupyter_notebooks/cfar/detection_result.png', dpi=120, bbox_inches='tight')
plt.show(); print('saved detection_result.png')